# One question, three tools

**Week 2, Thursday. The second case, in pairs, 45 minutes.** The brief is
`exercises/guided/C2_W02_D04_three_tools_STUDENT.md`. Replace each `__TODOn__` with the
option you choose; run as shipped, the notebook stops at `__TODO1__`.

> **The client's ask.** Kavya Nair, senior analyst: "You did the tree in plain Python in Week
> 1, in SQL on Monday. Do it a third way now, and tell me honestly which tool you would pick
> for which job."

The question all three tools answer is the tree node that moved in Week 1: **Retail-Plus
orders per customer, Q1 against Q2.**

In [ ]:
import sys, pathlib
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts")); break
import c2kit as kit
import pandas as pd
ENG = kit.engine()
print("connected to", kit.WAREHOUSE["dbname"])

In [ ]:
kit.side_by_side(
    kit.ladder(['Round 1: one row per customer', 'Round 2: the exposure merge', 'Round 3: the months view', 'Afternoon: the Monday table', 'Afternoon: three tools, one question'], lit=4, show=False),
    kit.flow(["plain Python\nevery step visible", "SQL\nwhere the data lives",
              "pandas\nthe analyst's bench"], title="Three tools, one number each", show=False),
)

## Step 1. Plain Python: the accumulator, every step visible

The rows arrive as a list of dictionaries, exactly the shape of Week 1's orders.

In [ ]:
rows = kit.sql('''SELECT o.order_id, o.customer_id, o.quarter FROM orders o
                  JOIN customers c ON c.customer_id = o.customer_id
                  WHERE c.segment = 'Retail-Plus' ''')
orders_n = {"Q1": 0, "Q2": 0}
# TODO 1. What holds each quarter's distinct customers?
#   a) a list, appended once per row
#   b) a count of rows, added once per row
#   c) a set, added to once per row
#   d) the length of rows
# Fill with: [], 0, set() or len(rows)
seen = {"Q1": __TODO1__, "Q2": __TODO1__}
for r in rows:
    orders_n[r["quarter"]] += 1
    seen[r["quarter"]].add(r["customer_id"])
py = {q: orders_n[q] / len(seen[q]) for q in ("Q1", "Q2")}
print("plain Python:", {q: round(v, 3) for q, v in py.items()})

In [ ]:
kit.check("plain Python counts 355 Retail-Plus orders", sum(orders_n.values()) == 355)
kit.check("Q1: 215 orders from 91 members", orders_n["Q1"] == 215 and len(seen["Q1"]) == 91)
kit.table(["quarter", "orders", "members", "orders per member"],
          [(q, orders_n[q], len(seen[q]), f"{py[q]:.3f}") for q in ("Q1", "Q2")], caption="Plain Python")

## Step 2. SQL: the same number, where the data lives

In [ ]:
# TODO 2. Which expression avoids Monday's integer-division trap?
#   a) count(*) / count(DISTINCT o.customer_id)
#   b) count(*)::numeric / count(DISTINCT o.customer_id)
#   c) avg(count(*))
#   d) count(DISTINCT o.order_id) / count(*)
# TODO 3. Where does the Retail-Plus filter go?
#   a) WHERE c.segment = 'Retail-Plus'
#   b) HAVING c.segment = 'Retail-Plus'
#   c) ORDER BY c.segment
#   d) LIMIT 120
SQL = '''SELECT o.quarter, count(*) AS orders, count(DISTINCT o.customer_id) AS members,
                round(__TODO2__, 3) AS per_member
         FROM orders o JOIN customers c ON c.customer_id = o.customer_id
         __TODO3__
         GROUP BY o.quarter ORDER BY o.quarter'''
sql_rows = kit.sql(SQL)
sq = {r["quarter"]: float(r["per_member"]) for r in sql_rows}
print("SQL:", sq)

In [ ]:
kit.check("SQL returns one row per quarter", [r["quarter"] for r in sql_rows] == ["Q1", "Q2"])
kit.check("SQL agrees with plain Python to three places", all(abs(sq[q] - py[q]) < 0.001 for q in sq), sq)

## Step 3. pandas: the analyst's bench

In [ ]:
df = pd.read_sql('''SELECT o.order_id, o.customer_id, o.quarter, c.segment FROM orders o
                    JOIN customers c ON c.customer_id = o.customer_id''', ENG)
# TODO 4. Which named aggregation counts members, each once?
#   a) members=("customer_id", "count")
#   b) members=("order_id", "nunique")
#   c) members=("customer_id", "size")
#   d) members=("customer_id", "nunique")
pd_view = (df[df["segment"] == "Retail-Plus"]
           .groupby("quarter")
           .agg(orders=("order_id", "count"), __TODO4__))
pd_view = pd_view.assign(per_member=pd_view["orders"] / pd_view["members"])
print(pd_view.round(3).to_string())

In [ ]:
pn = pd_view["per_member"].to_dict()
kit.check("pandas agrees with plain Python", all(abs(pn[q] - py[q]) < 1e-9 for q in pn))
kit.check("Retail-Plus orders per member fell from 2.363 to 1.842",
          round(pn["Q1"], 3) == 2.363 and round(pn["Q2"], 3) == 1.842)
kit.columns(["Q1", "Q2"], [("plain Python", [round(py[q], 3) for q in ("Q1", "Q2")]),
                           ("SQL", [sq[q] for q in ("Q1", "Q2")]),
                           ("pandas", [round(pn[q], 3) for q in ("Q1", "Q2")])],
            fmt=lambda v: f"{v:.3f}", width=640,
            title="Retail-Plus orders per member: three tools, one number")

## Step 4. The choice, with a reason per tool

The three agree, so the choice between them is about who has to trust, rerun or audit the
number. Finance's Monday revenue number is the one Kavya will ask you to defend.

In [ ]:
# TODO 5. Where should Finance's Monday revenue number be computed?
#   a) a pandas notebook on an analyst's laptop
#   b) a plain Python script with a loop
#   c) a SQL view in the warehouse
#   d) a spreadsheet exported every Monday
finance_home = __TODO5__
criteria = ["explain line by line", "audited where the data lives", "iterate fast on a question"]
kit.matrix(["plain Python", "SQL", "pandas"], criteria,
           [["best: every step visible", "no: runs off a copy", "slow to change"],
            ["reads as a statement", "best: Finance can rerun it", "fine for fixed asks"],
            ["reads as a chain", "no: runs off a copy", "best: the analyst's bench"]],
           title="The tool-choice note, as a grid")

In [ ]:
kit.check("the choice was made", finance_home is not None)
kit.check("the three tools agree on both quarters",
          all(abs(py[q] - pn[q]) < 1e-9 and abs(py[q] - sq[q]) < 0.001 for q in ("Q1", "Q2")))
fall = pn["Q2"] / pn["Q1"] - 1
kit.bridge(("Q1, orders per member", round(pn["Q1"], 3)), [("the fall", round(pn["Q2"] - pn["Q1"], 3))],
           end_label="Q2", fmt=lambda v: f"{v:.3f}", lit=(0,),
           title=f"Retail-Plus orders per member fell {abs(fall):.0%}: the lever Week 1 found")
kit.check_summary()

**Post:** your five letters, and your tool-choice note in the brief's format: one sentence
per tool, and the tool you would refuse for Finance's numbers, with the reason.